# 03 — Onset CNN (EXP-020), Colab / PyTorch

A fully-convolutional CNN predicts a per-frame **onset activation** from log-mel
frames; our **own** LFSF peak picker (`OnsetDetector._pick`) turns it into onset
times. No `librosa.onset.onset_detect`, no library peak-picker, no madmom — the
musical decision (peak picking) stays our code.

**Fair test first** (`TRAIN_ON_EXTRA_ONLY=True`): train on the 150
`train_extra_onsets` files, evaluate on the 127 main (c127, test-like) the model
never saw. The bar is fusion's c127 (0.8055, leakage-inflated) and the leaderboard
onset 0.775. If it clears that, retrain on all 277 (set the toggle False) for the
shippable model and we wire `OnsetDetector` after review.

Run in Colab (GPU). Upload `train.zip` + `train_extra_onsets.zip` to
`MyDrive/amp_data/`.


In [ ]:
# === Setup (Colab-ready) ===
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "librosa", "soundfile", "mir_eval"], check=False)
    REPO = Path("amp-challenge")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/8asic/amp2026-onset-beat-tempo.git",
                        str(REPO)], check=True)
else:
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))

import numpy as np
import librosa
import mir_eval
import torch
import torch.nn as nn

from src.config import config
from src.detectors import OnsetDetector
from src.utils import load_onsets_gt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "| repo:", REPO)

In [ ]:
# === Data: extract zips from Drive to fast local disk, then locate dirs ===
import zipfile

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_DIR = Path("/content/drive/MyDrive/amp_data")   # <-- folder with the zips
    WORK = Path("/content/data"); WORK.mkdir(exist_ok=True)
    for name in ["train", "train_extra_onsets"]:
        z, dest = ZIP_DIR / f"{name}.zip", WORK / name
        if dest.exists():
            print("already extracted:", dest); continue
        assert z.exists(), f"Missing {z} - upload {name}.zip to {ZIP_DIR}"
        print("extracting", z, "...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
    base = WORK
else:
    base = REPO / "data" / "processed"

def find_dir_with(suffix, root):
    root = Path(root)
    if not root.exists():
        return None
    for p in [root] + [d for d in root.rglob("*") if d.is_dir()]:
        if any(p.glob(f"*{suffix}")):
            return p
    return None

train_dir = find_dir_with(".onsets.gt", base / "train")
extra_dir = find_dir_with(".onsets.gt", base / "train_extra_onsets")
print("train_dir:", train_dir)
print("extra_dir:", extra_dir)
assert train_dir and extra_dir, "Could not locate .onsets.gt dirs - check ZIP_DIR / uploads"

In [ ]:
# === Collect onset-annotated files (127 main + 150 extra = 277) ===
def collect(d, corpus):
    d = Path(d); items = []
    if not d.exists():
        return items
    for wav in sorted(d.glob("*.wav")):
        gtp = d / f"{wav.stem}.onsets.gt"
        if gtp.exists():
            ons = load_onsets_gt(gtp)
            if ons is not None and len(ons):
                items.append((str(wav), np.asarray(ons, dtype=float), corpus))
    return items

main_files = collect(train_dir, "c127")          # 127 main (test-like)
extra_files = collect(extra_dir, "extra")        # 150 supplementary
files = main_files + extra_files
print(f"main(c127): {len(main_files)}  extra: {len(extra_files)}  total: {len(files)}")
assert files, "No files found - check DATA_ROOT path."

In [ ]:
# === Features (log-mel, onset config) + per-frame onset labels (cached) ===
SR = config.audio.sample_rate          # 22050
HOP = config.audio.onset_hop_length    # 256
NFFT = config.audio.onset_fft_size     # 2048
NMELS = config.audio.onset_n_mels      # 82
FMIN, FMAX = config.audio.onset_fmin, config.audio.onset_fmax
FPS = SR / HOP
LABEL_W = 1                            # +-frames around a GT onset labelled positive

def logmel(y):
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=NFFT, hop_length=HOP,
                                       n_mels=NMELS, fmin=FMIN, fmax=FMAX)
    return np.log1p(m).T.astype(np.float32)        # (T, NMELS)

def onset_labels(onsets, T):
    lab = np.zeros(T, dtype=np.float32)
    for f in np.round(onsets * FPS).astype(int):
        for dd in range(-LABEL_W, LABEL_W + 1):
            if 0 <= f + dd < T:
                lab[f + dd] = 1.0
    return lab

CACHE = REPO / "experiments" / ".cache_onset"
CACHE.mkdir(parents=True, exist_ok=True)

data = []
for i, (wav, onsets, corpus) in enumerate(files):
    stem = Path(wav).stem
    cp = CACHE / f"{stem}.npz"
    if cp.exists():
        d = np.load(cp)
        X, y = d["X"], d["y"]
    else:
        y_audio, _ = librosa.load(wav, sr=SR)
        X = logmel(y_audio)
        y = onset_labels(onsets, X.shape[0])
        np.savez(cp, X=X, y=y)
    data.append({"stem": stem, "X": X, "y": y, "onsets": onsets, "wav": wav, "corpus": corpus})
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(files)} features")
print("features ready:", len(data), "| n_mels", NMELS, "| fps", round(FPS, 2))

In [ ]:
# === Split + standardization (stats from TRAIN only) ===
# FAIR TEST: train on 150 extra_onsets, eval on 127 c127 (never trained on).
# Set False for the shippable model trained on all 277.
TRAIN_ON_EXTRA_ONLY = True

if TRAIN_ON_EXTRA_ONLY:
    train_set = [d for d in data if d["corpus"] == "extra"]
    val_set   = [d for d in data if d["corpus"] == "c127"]
else:
    rng = np.random.default_rng(0)
    order = rng.permutation(len(data))
    n_val = int(0.2 * len(data))
    val_ids = set(order[:n_val].tolist())
    train_set = [data[i] for i in range(len(data)) if i not in val_ids]
    val_set = [data[i] for i in range(len(data)) if i in val_ids]

print(f"train {len(train_set)}  val {len(val_set)}  (extra_only={TRAIN_ON_EXTRA_ONLY})")
assert train_set and val_set, "empty split"

allX = np.concatenate([d["X"] for d in train_set])
MU = allX.mean(0); SD = allX.std(0) + 1e-8
def norm(X):
    return (X - MU) / SD

In [ ]:
# === Dataset / loaders (batch_size=1: variable-length sequences) ===
class OnsetDS(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        d = self.items[i]
        return (torch.from_numpy(norm(d["X"])),     # (T, NMELS)
                torch.from_numpy(d["y"]))           # (T,)

train_dl = torch.utils.data.DataLoader(OnsetDS(train_set), batch_size=1, shuffle=True)
val_dl = torch.utils.data.DataLoader(OnsetDS(val_set), batch_size=1, shuffle=False)

In [ ]:
# === Model: fully-convolutional onset CNN -> per-frame activation ===
# Convs preserve the time axis (pool only over frequency), so the output is one
# activation per frame with a ~7-frame receptive field (good for local onsets).
class OnsetCNN(nn.Module):
    def __init__(self, n_mels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, (3, 3), padding=(1, 1)), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d((1, 3)),
            nn.Conv2d(16, 32, (3, 3), padding=(1, 1)), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((1, 3)),
            nn.Conv2d(32, 32, (3, 3), padding=(1, 1)), nn.BatchNorm2d(32), nn.ReLU(),
        )
        with torch.no_grad():
            f = self.conv(torch.zeros(1, 1, 8, n_mels)).shape[-1]
        self.head = nn.Sequential(
            nn.Linear(32 * f, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):                 # x: (B, T, n_mels)
        h = self.conv(x.unsqueeze(1))     # (B, 32, T, f)
        B, C, T, F = h.shape
        h = h.permute(0, 2, 1, 3).reshape(B, T, C * F)
        return self.head(h).squeeze(-1)   # (B, T) logits

model = OnsetCNN(NMELS).to(DEVICE)
print(sum(p.numel() for p in model.parameters()), "params")

In [ ]:
# === Train (BCE pos-weighted + grad-clip + ReduceLROnPlateau + early stop) ===
POS_WEIGHT = torch.tensor([4.0], device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
EPOCHS, ES_PATIENCE = 60, 8

best_val, best_state, since_best = float("inf"), None, 0
for ep in range(1, EPOCHS + 1):
    model.train(); tr = 0.0
    for X, y in train_dl:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = crit(model(X), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        opt.step()
        tr += loss.item()
    model.eval(); va = 0.0
    with torch.no_grad():
        for X, y in val_dl:
            X, y = X.to(DEVICE), y.to(DEVICE)
            va += crit(model(X), y).item()
    va /= len(val_dl)
    sched.step(va)
    if va < best_val:
        best_val, since_best = va, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        since_best += 1
    print(f"epoch {ep:2d}  train {tr/len(train_dl):.4f}  val {va:.4f}  "
          f"lr {opt.param_groups[0]['lr']:.2e}")
    if since_best >= ES_PATIENCE:
        print(f"early stop at epoch {ep}"); break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"best val loss {best_val:.4f}")

In [ ]:
# === Decode with OUR peak picker; eval by corpus vs fusion baseline ===
od = OnsetDetector()

@torch.no_grad()
def activation(X):
    model.eval()
    t = torch.from_numpy(norm(X)).unsqueeze(0).to(DEVICE)
    return torch.sigmoid(model(t)).squeeze(0).cpu().numpy()

def f1_at(acts, delta):
    fs = {"c127": [], "extra": []}
    for d, a in zip(val_set, acts):
        peaks = od._pick(np.asarray(a, dtype=np.float64), FPS, delta)
        est = np.array(peaks, dtype=int) / FPS
        f = mir_eval.onset.f_measure(d["onsets"], est, window=0.05)[0] if len(est) else 0.0
        fs[d["corpus"]].append(f)
    return fs

acts = [activation(d["X"]) for d in val_set]
print("CNN onset F1 by peak-pick delta (val):")
best = (None, -1.0)
for delta in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    fs = f1_at(acts, delta)
    allf = fs["c127"] + fs["extra"]
    m = float(np.mean(allf))
    c127 = float(np.mean(fs["c127"])) if fs["c127"] else 0.0
    print(f"  delta={delta:.2f}: all={m:.4f}  c127={c127:.4f} ({len(fs['c127'])})  "
          f"extra={np.mean(fs['extra']) if fs['extra'] else 0:.4f}")
    if c127 > best[1]:
        best = (delta, c127)
print(f"\nBEST c127: delta={best[0]}  F1={best[1]:.4f}")

# Fusion baseline (current EXP-015 onset) on the SAME val files
config.onset.learned = False           # ensure pure-numpy LR off
od_fus = OnsetDetector()
fus = {"c127": [], "extra": []}
for d in val_set:
    y_audio, _ = librosa.load(d["wav"], sr=SR)
    est = od_fus.detect(y_audio, SR)
    f = mir_eval.onset.f_measure(d["onsets"], np.asarray(est), window=0.05)[0] if len(est) else 0.0
    fus[d["corpus"]].append(f)
print(f"FUSION baseline  c127={np.mean(fus['c127']):.4f}  "
      f"extra={np.mean(fus['extra']) if fus['extra'] else 0:.4f}")
print("Bar: beat fusion on c127 (~0.806) and leaderboard onset 0.775.")

In [ ]:
# === Save weights + everything inference needs ===
fname = "onset_cnn_extra.pt" if TRAIN_ON_EXTRA_ONLY else "onset_cnn.pt"
out = REPO / "models" / fname
out.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "state_dict": model.state_dict(),
    "mu": MU, "sd": SD,
    "n_mels": NMELS, "sr": SR, "hop": HOP, "n_fft": NFFT, "fmin": FMIN, "fmax": FMAX,
    "label_w": LABEL_W,
}, out)
print("saved", out)
# In Colab: from google.colab import files; files.download(str(out))